# Deepfake Detector v5 - Dataset Combinado

Combina múltiplos datasets para melhor generalização:
- **Fakes**: manjilkarki/deepfake-and-real-images (face-swap)
- **Reais**: LFW (Labeled Faces in the Wild) + fotos reais do manjilkarki

Isso resolve o viés de classificar selfies/fotos de celular como fake.

In [ ]:
# Instalar dependências
!pip install -q torch torchvision timm scikit-learn pillow tqdm kagglehub

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, f1_score, precision_recall_curve
import timm
from tqdm.auto import tqdm
import urllib.request
import tarfile
import kagglehub

# Seed para reprodutibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando: {device}')

## 1. Download dos Datasets

In [ ]:
# Dataset 1: manjilkarki (deepfakes + reais de vídeo)
print("Baixando dataset manjilkarki...")
manjil_path = kagglehub.dataset_download("manjilkarki/deepfake-and-real-images")
print(f"Dataset manjilkarki em: {manjil_path}")

In [ ]:
# Dataset 2: LFW via sklearn (mais confiável que download direto)
# Alternativa: usar dataset do Kaggle
from sklearn.datasets import fetch_lfw_people

print("Baixando LFW dataset via sklearn...")
try:
    # Tenta via sklearn primeiro
    lfw_data = fetch_lfw_people(min_faces_per_person=1, resize=1.0, download_if_missing=True)
    LFW_PATH = os.path.join(os.path.expanduser('~'), 'scikit_learn_data', 'lfw_home', 'lfw_funneled')
    print(f"LFW baixado via sklearn em: {LFW_PATH}")
except Exception as e:
    print(f"Sklearn falhou: {e}")
    print("Tentando via Kaggle...")
    # Fallback: usar dataset CelebA crop do Kaggle (faces reais diversas) Celeb-DF v2
    LFW_PATH = kagglehub.dataset_download("jessicali9530/celeba-dataset")
    # Usar apenas as imagens de face
    LFW_PATH = os.path.join(LFW_PATH, "img_align_celeba", "img_align_celeba")
    print(f"CelebA baixado em: {LFW_PATH}")

In [ ]:
def collect_images(root_dir, extensions=('.jpg', '.jpeg', '.png')):
    """Coleta todos os caminhos de imagem recursivamente."""
    images = []
    for root, dirs, files in os.walk(root_dir):
        for f in files:
            if f.lower().endswith(extensions):
                images.append(os.path.join(root, f))
    return images

# Coletar imagens do manjilkarki
manjil_fake_train = collect_images(os.path.join(manjil_path, 'Dataset/Train/Fake'))
manjil_fake_test = collect_images(os.path.join(manjil_path, 'Dataset/Test/Fake'))
manjil_real_train = collect_images(os.path.join(manjil_path, 'Dataset/Train/Real'))
manjil_real_test = collect_images(os.path.join(manjil_path, 'Dataset/Test/Real'))

# Coletar imagens do LFW
lfw_images = collect_images(LFW_PATH)

print(f"Manjilkarki - Fake train: {len(manjil_fake_train)}, test: {len(manjil_fake_test)}")
print(f"Manjilkarki - Real train: {len(manjil_real_train)}, test: {len(manjil_real_test)}")
print(f"LFW - Total: {len(lfw_images)}")

In [ ]:
# Balancear os datasets
# Estratégia: usar todos os fakes, combinar reais de ambos datasets

all_fakes = manjil_fake_train + manjil_fake_test
all_reals_manjil = manjil_real_train + manjil_real_test

# Quantos reais precisamos para balancear?
n_fakes = len(all_fakes)
print(f"Total de fakes: {n_fakes}")

# Usar metade de manjilkarki (reais de vídeo) e metade de LFW (fotos variadas)
n_real_each = n_fakes // 2

# Amostrar de cada fonte
random.shuffle(all_reals_manjil)
random.shuffle(lfw_images)

reals_from_manjil = all_reals_manjil[:n_real_each]
reals_from_lfw = lfw_images[:n_real_each]

all_reals = reals_from_manjil + reals_from_lfw
random.shuffle(all_reals)

print(f"Reais de manjilkarki: {len(reals_from_manjil)}")
print(f"Reais de LFW: {len(reals_from_lfw)}")
print(f"Total reais: {len(all_reals)}")
print(f"Ratio fake/real: {n_fakes/len(all_reals):.2f}")

In [ ]:
# Criar dataset combinado
# Label: 1 = FAKE, 0 = REAL

all_images = [(p, 1) for p in all_fakes] + [(p, 0) for p in all_reals]
random.shuffle(all_images)

# Split 80/10/10 (train/val/test)
train_data, temp_data = train_test_split(all_images, test_size=0.2, random_state=SEED, stratify=[x[1] for x in all_images])
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=SEED, stratify=[x[1] for x in temp_data])

print(f"Train: {len(train_data)} ({sum(x[1] for x in train_data)} fakes)")
print(f"Val: {len(val_data)} ({sum(x[1] for x in val_data)} fakes)")
print(f"Test: {len(test_data)} ({sum(x[1] for x in test_data)} fakes)")

## 2. Dataset e Transforms

In [ ]:
class DeepfakeDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data = data
        self.transform = transform
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label
        except Exception as e:
            # Em caso de imagem corrompida, retorna outra
            new_idx = (idx + 1) % len(self.data)
            return self.__getitem__(new_idx)

# Transforms com augmentation mais agressivo para generalização
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    # Simular compressão JPEG (comum em fotos de celular)
    transforms.RandomApply([transforms.GaussianBlur(3)], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.1),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Criar datasets
train_dataset = DeepfakeDataset(train_data, train_transform)
val_dataset = DeepfakeDataset(val_data, val_transform)
test_dataset = DeepfakeDataset(test_data, val_transform)

print(f"Datasets criados: train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}")

In [ ]:
# DataLoaders
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

## 3. Modelo

In [ ]:
class DeepfakeDetectorV5(nn.Module):
    """Modelo v5 com regularização extra para generalização."""
    
    def __init__(self, model_name='efficientnet_b0', dropout=0.5):
        super().__init__()
        
        # Backbone com pretrained weights
        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0,
            global_pool='avg'
        )
        
        num_features = self.backbone.num_features
        
        # Classifier com mais regularização
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(num_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.5),
            nn.Linear(256, 1)
        )
    
    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features).squeeze(-1)

model = DeepfakeDetectorV5(dropout=0.5).to(device)
print(f"Modelo criado: {sum(p.numel() for p in model.parameters()):,} parâmetros")

## 4. Treinamento

In [ ]:
# Focal Loss para lidar com exemplos difíceis
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.5, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, inputs, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(inputs, targets.float(), reduction='none')
        pt = torch.exp(-bce)
        focal_weight = (1 - pt) ** self.gamma
        
        # Alpha balancing
        alpha_weight = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        
        return (alpha_weight * focal_weight * bce).mean()

criterion = FocalLoss(alpha=0.5, gamma=2.0)  # alpha=0.5 para dataset balanceado
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc='Training')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item()
        preds = (torch.sigmoid(outputs) > 0.5).long()
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
        pbar.set_postfix({'loss': loss.item(), 'acc': correct/total})
    
    return total_loss / len(loader), correct / total

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    total_loss = 0
    
    for images, labels in tqdm(loader, desc='Evaluating'):
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        total_loss += loss.item()
        
        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).long()
        
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    acc = (all_preds == all_labels).mean()
    auc = roc_auc_score(all_labels, all_probs)
    f1 = f1_score(all_labels, all_preds)
    
    return total_loss / len(loader), acc, auc, f1, all_probs, all_labels

In [ ]:
# Treinamento
EPOCHS = 15
best_auc = 0
best_model_state = None
patience = 5
no_improve = 0

for epoch in range(EPOCHS):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"{'='*50}")
    
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc, val_auc, val_f1, _, _ = evaluate(model, val_loader)
    
    scheduler.step()
    
    print(f"Train - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}")
    print(f"Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, AUC: {val_auc:.4f}, F1: {val_f1:.4f}")
    
    if val_auc > best_auc:
        best_auc = val_auc
        best_model_state = model.state_dict().copy()
        print(f"✓ Novo melhor modelo! AUC: {best_auc:.4f}")
        no_improve = 0
    else:
        no_improve += 1
        print(f"Sem melhora ({no_improve}/{patience})")
    
    if no_improve >= patience:
        print("Early stopping!")
        break

# Carregar melhor modelo
model.load_state_dict(best_model_state)

## 5. Calibração de Threshold

In [ ]:
# Avaliar no conjunto de validação para calibrar threshold
_, _, _, _, val_probs, val_labels = evaluate(model, val_loader)

# Encontrar threshold que minimiza falsos positivos (fotos reais como fake)
precision, recall, thresholds = precision_recall_curve(val_labels, val_probs)

# Calcular F1 para cada threshold
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5

# Também calcular threshold conservador (alta precisão)
# Queremos precisão > 0.9 para minimizar falsos positivos
high_precision_idx = np.where(precision >= 0.9)[0]
if len(high_precision_idx) > 0:
    conservative_threshold = thresholds[high_precision_idx[0]] if high_precision_idx[0] < len(thresholds) else 0.7
else:
    conservative_threshold = 0.7

print(f"Threshold ótimo (F1): {best_threshold:.4f}")
print(f"Threshold conservador (precisão ≥90%): {conservative_threshold:.4f}")

# Usar threshold conservador para evitar falsos positivos
final_threshold = conservative_threshold
print(f"\nUsando threshold: {final_threshold:.4f}")

## 6. Avaliação Final

In [ ]:
# Avaliar no conjunto de teste com threshold calibrado
test_loss, test_acc, test_auc, test_f1, test_probs, test_labels = evaluate(model, test_loader)

# Calcular métricas com threshold conservador
test_preds_calibrated = (test_probs > final_threshold).astype(int)

print("\n" + "="*50)
print("RESULTADOS FINAIS (Conjunto de Teste)")
print("="*50)
print(f"AUC: {test_auc:.4f}")
print(f"Threshold: {final_threshold:.4f}")
print("\nCom threshold padrão (0.5):")
print(f"  Accuracy: {test_acc:.4f}")
print(f"  F1-Score: {test_f1:.4f}")
print(f"\nCom threshold conservador ({final_threshold:.2f}):")
calibrated_acc = (test_preds_calibrated == test_labels).mean()
calibrated_f1 = f1_score(test_labels, test_preds_calibrated)
print(f"  Accuracy: {calibrated_acc:.4f}")
print(f"  F1-Score: {calibrated_f1:.4f}")

print("\nRelatório detalhado:")
print(classification_report(test_labels, test_preds_calibrated, target_names=['REAL', 'FAKE']))

In [ ]:
# Analisar taxas de erro por fonte
print("\n" + "="*50)
print("ANÁLISE DE FALSOS POSITIVOS")
print("="*50)

# Separar reais e fakes do teste
real_indices = [i for i, (path, label) in enumerate(test_data) if label == 0]
fake_indices = [i for i, (path, label) in enumerate(test_data) if label == 1]

real_probs = test_probs[real_indices]
fake_probs = test_probs[fake_indices]

print(f"\nImagens REAIS:")
print(f"  Média de p_fake: {real_probs.mean():.4f}")
print(f"  Mediana de p_fake: {np.median(real_probs):.4f}")
print(f"  Max p_fake: {real_probs.max():.4f}")
print(f"  Falsos positivos (>{final_threshold:.2f}): {(real_probs > final_threshold).sum()} de {len(real_probs)} ({100*(real_probs > final_threshold).mean():.1f}%)")

print(f"\nImagens FAKE:")
print(f"  Média de p_fake: {fake_probs.mean():.4f}")
print(f"  Mediana de p_fake: {np.median(fake_probs):.4f}")
print(f"  Min p_fake: {fake_probs.min():.4f}")
print(f"  Falsos negativos (<{final_threshold:.2f}): {(fake_probs < final_threshold).sum()} de {len(fake_probs)} ({100*(fake_probs < final_threshold).mean():.1f}%)")

## 7. Salvar Modelo

In [ ]:
# Salvar checkpoint completo
checkpoint = {
    'model_state_dict': best_model_state,
    'best_auc': best_auc,
    'best_threshold': float(final_threshold),
    'test_auc': float(test_auc),
    'test_f1': float(calibrated_f1),
    'config': {
        'model_name': 'efficientnet_b0',
        'version': 'v5',
        'dropout': 0.5,
        'datasets': ['manjilkarki/deepfake-and-real-images', 'LFW'],
        'description': 'Treinado com dataset combinado para melhor generalização'
    }
}

save_path = 'efficientnet_b0_deepfake_v5.pth'
torch.save(checkpoint, save_path)
print(f"\n✓ Modelo salvo em: {save_path}")
print(f"  AUC: {best_auc:.4f}")
print(f"  Threshold: {final_threshold:.4f}")

In [ ]:
# Instruções para usar o modelo
print("""
╔══════════════════════════════════════════════════════════════════╗
║                    MODELO TREINADO COM SUCESSO!                 ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  Para usar na aplicação:                                         ║
║                                                                  ║
║  1. Copie o arquivo para a pasta de pesos:                       ║
║     cp efficientnet_b0_deepfake_v5.pth \\                         ║
║        ~/Documentos/detectar/model/weights/                      ║
║                                                                  ║
║  2. Reinicie o worker para carregar o novo modelo                ║
║                                                                  ║
║  O modelo v5 inclui:                                             ║
║  - Dataset combinado (manjilkarki + LFW)                         ║
║  - Melhor generalização para selfies/fotos de celular            ║
║  - Threshold conservador para menos falsos positivos             ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
""")